# Agent —— 智能代理

> Agent 让 LLM 自主决定调用哪些工具、按什么顺序调用，直到完成任务。
> 最新推荐使用 `langchain.agents.create_agent`（LangChain v1.0+）。

## 1. 准备工作：定义工具

> Agent 的核心是「工具」，先定义几个演示工具。

In [ ]:
from langchain_core.tools import tool
from rich import print as rprint

@tool
def calculate(expression: str) -> str:
    """计算数学表达式，输入为数学表达式字符串，如 '(3 + 5) * 2'。"""
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"计算错误：{e}"

@tool
def get_weather(city: str) -> str:
    """根据城市名称查询当前天气信息。"""
    mock_weather = {
        "北京": "晴天，25°C",
        "上海": "多云，22°C",
        "广州": "小雨，28°C",
        "深圳": "晴，30°C",
    }
    return mock_weather.get(city, f"未找到{city}的天气信息")

@tool
def get_current_time() -> str:
    """获取当前时间。"""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

tools = [calculate, get_weather, get_current_time]
rprint(f"已定义 {len(tools)} 个工具：{[t.name for t in tools]}")

已定义 3 个工具：['calculate', 'get_weather', 'get_current_time']

## 2. create_agent：最简洁的 Agent 创建方式（推荐）

> `from langchain.agents import create_agent`
> 只需 `model` + `tools` + `system_prompt` 三个参数，无需手动构造 prompt 模板。
> 底层基于 LangGraph，支持持久化、中间件等高级特性。

In [4]:
import os
import dotenv
from langchain.agents import create_agent

dotenv.load_dotenv()

# create_agent：三行代码创建 Agent
# model 可以传字符串 "provider:model_name" 或已初始化的模型实例
agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个有帮助的助手，可以使用工具来回答问题。",
)

# invoke 时传入 messages 列表（OpenAI 消息格式）
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "现在几点了？北京天气怎么样？再算一下 (25+15)*3"},
    ]
})

# 最终回答在 messages 的最后一条
print(f"最终回答：{result['messages'][-1]['content']}")

TypeError: 'AIMessage' object is not subscriptable

### 2.1 查看 Agent 的完整执行过程

> `result['messages']` 包含完整的对话历史，包括工具调用和工具返回结果。

In [5]:
import os
import dotenv
from langchain.agents import create_agent
from rich import print as rprint

dotenv.load_dotenv()

agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个有帮助的助手。",
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "北京和上海的天气分别怎么样？"}],
})

rprint(f"消息总数：{len(result['messages'])}")
for i, msg in enumerate(result['messages']):
    role = msg.get("role", msg.get("type", "unknown"))
    content = str(msg.get("content", ""))[:80]
    rprint(f"  [{i}] {role}: {content}")

消息总数：5

AttributeError: 'HumanMessage' object has no attribute 'get'

## 3. create_agent + 多轮对话（thread_id 持久化）

> 通过 `checkpointer` 和 `thread_id` 自动持久化对话历史。
> 同一 `thread_id` 的多次 invoke 共享上下文，Agent 能记住之前聊过什么。

In [ ]:
import os
import dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from rich import print as rprint

dotenv.load_dotenv()

# MemorySaver 作为 checkpointer，负责保存对话状态
checkpointer = MemorySaver()

agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个有帮助的助手。记住用户的信息，在后续对话中可以用到。",
    checkpointer=checkpointer,   # ← 启用持久化
)

# ── 第一轮 ──
r1 = agent.invoke({
    "messages": [{"role": "user", "content": "你好，我叫小明，来自北京"}],
    "configurable": {"thread_id": "user-001"},  # 标记同一对话
})
rprint(f"用户：你好，我叫小明，来自北京")
rprint(f"Agent：{r1['messages'][-1]['content']}\n")

# ── 第二轮（同一 thread_id，Agent 记住了） ──
r2 = agent.invoke({
    "messages": [{"role": "user", "content": "我叫什么名字？我来自哪个城市？那边天气怎么样？"}],
    "configurable": {"thread_id": "user-001"},
})
rprint(f"用户：我叫什么名字？我来自哪个城市？那边天气怎么样？")
rprint(f"Agent：{r2['messages'][-1]['content']}")

## 4. create_agent + response_format（结构化输出）

> `response_format` 参数原生支持结构化输出，Agent 查完工具后直接返回 Pydantic 模型。

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from rich import print as rprint

dotenv.load_dotenv()

class WeatherReport(BaseModel):
    city: str = Field(description="城市")
    temperature: str = Field(description="温度")
    condition: str = Field(description="天气状况")
    suggestion: str = Field(description="出行建议")

# response_format 直接传入 Pydantic 模型
agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个天气助手，先查天气数据再回答。",
    response_format=WeatherReport,
    checkpointer=MemorySaver(),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "上海天气怎么样？"}],
    "configurable": {"thread_id": "struct-001"},
})

# 当设置了 response_format，最终消息的 content 是 Pydantic 模型实例
last_msg = result['messages'][-1]
# content 可能已经是 dict（序列化后），也可能是模型实例
if isinstance(last_msg.get('content'), dict):
    report = WeatherReport.model_validate(last_msg['content'])
else:
    report = last_msg['content']

rprint(f"类型：{type(report).__name__}")
rprint(f"城市：{report.city}")
rprint(f"温度：{report.temperature}")
rprint(f"天气：{report.condition}")
rprint(f"建议：{report.suggestion}")

## 5. create_agent + response_format：多城市对比

> Agent 可以连续调用多次工具，最终一次性输出结构化的对比报告。

In [ ]:
import os
import dotenv
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from rich import print as rprint

dotenv.load_dotenv()

class CityWeather(BaseModel):
    """单个城市的天气。"""
    city: str = Field(description="城市名称")
    weather: str = Field(description="天气描述")
    temperature: str = Field(description="温度")

class MultiCityReport(BaseModel):
    """多城市对比报告。"""
    cities: list[CityWeather] = Field(description="各城市天气列表")
    best_city: str = Field(description="天气最好的城市")
    summary: str = Field(description="综合总结")

agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个天气分析助手。先查询各城市的天气，再给出对比报告。",
    response_format=MultiCityReport,
    checkpointer=MemorySaver(),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "查询北京、上海、广州三地天气，对比分析哪个最好"}],
    "configurable": {"thread_id": "multi-001"},
})

last_msg = result['messages'][-1]
if isinstance(last_msg.get('content'), dict):
    report = MultiCityReport.model_validate(last_msg['content'])
else:
    report = last_msg['content']

rprint(report.model_dump())

## 6. create_agent + 中间件（Middleware）

> 中间件是 `create_agent` 的特色功能，可以在 Agent 执行的不同阶段插入自定义逻辑。
> 例如：敏感信息过滤、对话摘要、人工审核等。

In [ ]:
import os
import dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langgraph.checkpoint.memory import MemorySaver
from rich import print as rprint

dotenv.load_dotenv()

# PIIMiddleware：自动过滤敏感信息（手机号、邮箱等）
agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个助手。",
    middleware=[
        PIIMiddleware(
            # 检测手机号并脱敏
            detector="phone_number",
            strategy="redact",
            apply_to_input=True,
            apply_to_output=True,
        ),
    ],
    checkpointer=MemorySaver(),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "我的手机号是13800138000，查一下北京的天气"}],
    "configurable": {"thread_id": "pii-001"},
})

rprint(f"最终回答：{result['messages'][-1]['content']}")

## 7. 自定义 Agent System Prompt

> `system_prompt` 参数接受字符串或 `SystemMessage`，可以定制 Agent 的个性和行为。

In [ ]:
import os
import dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

dotenv.load_dotenv()

# 定制角色：幽默风趣的天气小诸葛
agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt=(
        "你是一个幽默风趣的天气预报员，外号「天气小诸葛」。\n"
        "你非常热爱用比喻和段子来描述天气，每次回答都要让人会心一笑。\n"
        "但是数据必须准确，先用工具查了再说。\n"
        "回答控制在 100 字以内。"
    ),
    checkpointer=MemorySaver(),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "北京和广州天气咋样？"}],
    "configurable": {"thread_id": "humor-001"},
})

print(f"『天气小诸葛』说：{result['messages'][-1]['content']}")

## 8. 限制 Agent 最大迭代次数

> Agent 可能会无限循环，使用 `interrupt_before` 或 LangGraph 的 `recursion_limit` 限制。

In [ ]:
import os
import dotenv
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

dotenv.load_dotenv()

agent = create_agent(
    model="openai:" + os.getenv("MODEL_NAME"),
    tools=tools,
    system_prompt="你是一个助手。",
    checkpointer=MemorySaver(),
)

# 通过 config 的 recursion_limit 限制最大迭代次数
result = agent.invoke({
    "messages": [{"role": "user", "content": "查北京、上海、广州、深圳四个城市的天气，再计算温度总和"}],
    "configurable": {"thread_id": "limit-001"},
    "recursion_limit": 20,  # ← 限制最大 20 步
})

print(f"最终回答：{result['messages'][-1]['content']}")

## 9. 使用 ChatOpenAI 实例创建 Agent

> `model` 参数可以直接传已初始化的 ChatOpenAI 实例，方便传递自定义参数。

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

dotenv.load_dotenv()

# 先初始化模型，再传给 create_agent
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0,
)

agent = create_agent(
    model=llm,      # ← 直接传模型实例
    tools=tools,
    system_prompt="你是一个有帮助的助手。",
    checkpointer=MemorySaver(),
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "深圳天气怎么样？现在几点了？"}],
    "configurable": {"thread_id": "llm-001"},
})

print(f"最终回答：{result['messages'][-1]['content']}")

## 10. 传统方式（参考）：create_react_agent

> `create_react_agent` 是旧版 API，`create_agent` 是其替代品。
> 如果你的 LangChain 版本低于 v1.0，可以参考这种方式。

In [ ]:
# ── 以下代码仅作参考（需要 langgraph 早期版本） ──
# 
# from langgraph.prebuilt import create_react_agent
# 
# agent = create_react_agent(
#     model=llm,
#     tools=tools,
#     prompt="你是一个有帮助的助手。",
# )
# 
# result = agent.invoke({
#     "messages": [{"role": "user", "content": "北京天气怎么样？"}]
# })
# print(result["messages"][-1].content)

print("当前使用 LangChain v1.3.7，推荐使用 create_agent")

---

## 总结：create_agent vs 旧 API

| 对比维度 | create_agent ✅ | create_tool_calling_agent ❌ | create_react_agent ❌ |
|---------|----------------|---------------------------|---------------------|
| 所属包 | `langchain.agents` | `langchain.agents`（v1.0 已移除） | `langgraph.prebuilt` |
| 参数 | `model + tools + system_prompt` 简单三参数 | 需要手动构造 ChatPromptTemplate + MessagesPlaceholder | `model + tools + prompt` |
| prompt | 自动处理 | 用户手动拼接 | 用户手动拼接 |
| 持久化 | ✅ checkpointer + thread_id | ❌ 需手动维护 chat_history | ✅ checkpointer（langgraph） |
| 结构化输出 | ✅ response_format 原生支持 | ❌ 需额外 with_structured_output | ❌ 需额外处理 |
| 中间件 | ✅ PIIMiddleware 等开箱即用 | ❌ | ❌ |
| 返回格式 | OpenAI 消息字典列表 | AgentExecutor 返回 dict | 消息对象列表 |
| 推荐度 | ⭐⭐⭐ 强烈推荐 | 已废弃 | 已废弃 |

In [ ]:
from rich import print as rprint

rprint("[bold green]✅ create_agent 是最新推荐方式，已覆盖全部常用场景：[/bold green]")
rprint("  1. 基础调用：model + tools + system_prompt")
rprint("  2. 多轮记忆：checkpointer + thread_id")
rprint("  3. 结构化输出：response_format")
rprint("  4. 中间件：PIIMiddleware 等")
rprint("  5. 自定义：ChatOpenAI 实例传参")